# Integração entre a API IBGE e BC

Este notebook utiliza o código de API_IBGE e API_BC rodando-os com o %run a fim de rodar ambos os códigos e integrá-los em uma tabela única.

## Importando os módulos necessários

In [1]:
import hashlib
import json
import unicodedata
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urlencode
import numpy as np
import altair as alt
import pandas as pd
import requests
import requests as rqs
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

### Rodando o código de API_IBGE

In [9]:
%run API_IBGE.ipynb

ConnectTimeout: HTTPSConnectionPool(host='servicodados.ibge.gov.br', port=443): Max retries exceeded with url: /api/v3/agregados/1612/metadados (Caused by ConnectTimeoutError(<HTTPSConnection(host='servicodados.ibge.gov.br', port=443) at 0x2438869e990>, 'Connection to servicodados.ibge.gov.br timed out. (connect timeout=30)'))

ConnectTimeout: HTTPSConnectionPool(host='servicodados.ibge.gov.br', port=443): Max retries exceeded with url: /api/v3/agregados/1612/metadados (Caused by ConnectTimeoutError(<HTTPSConnection(host='servicodados.ibge.gov.br', port=443) at 0x2438869e990>, 'Connection to servicodados.ibge.gov.br timed out. (connect timeout=30)'))

### Tratando API_BC

ASDADSAS

In [4]:
CHAVES = ["codigo_municipio", "cultura", "ano"]
cultura_por_id = {"2713": "soja", "2711": "milho", "2696": "cana_de_acucar"}

medidas = {
    "area_plantada_ha": df_area_municipal,
    "producao_t": df_prod_municipal,
    "rendimento_kg_ha": df_rendimento_municipal,
}

ibge = None
nomes = []

for nome, df in medidas.items():
    df = df.copy()
    df["cultura"] = df["codigo_cultura"].astype(str).map(cultura_por_id)
    df["codigo_municipio"] = df["codigo_municipio"].astype("string").str.strip()
    df["ano"] = df["ano"].astype("int64")
    
    nomes.append(df[["codigo_municipio", "municipio"]])
    parte = df[CHAVES + ["valor"]].rename(columns={"valor": nome})
    ibge = parte if ibge is None else ibge.merge(parte, on=CHAVES, how="outer")

nomes = pd.concat(nomes).drop_duplicates("codigo_municipio")



NameError: name 'df_area_municipal' is not defined

In [ ]:
# Se você já tem 'credito_anual' em memória ou em CSV:
# credito = pd.read_csv("dados/credito_anual.csv", dtype={"codigo_municipio": str})
credito = credito_anual[CHAVES + ["credito_nominal_brl"]].copy()
credito["codigo_municipio"] = credito["codigo_municipio"].astype("string").str.strip()
credito["ano"] = credito["ano"].astype("int64")

# Realiza o Merge (outer join)
tabela = ibge.merge(credito, on=CHAVES, how="outer", indicator="cobertura")

# Mapeia a coluna de cobertura para indicar a origem dos dados
tabela["cobertura"] = tabela["cobertura"].astype(str).map({
    "both": "ambos",
    "left_only": "so_ibge",
    "right_only": "so_credito"
})

# Adiciona o nome do município e estado (UF)
tabela = tabela.merge(nomes, on="codigo_municipio", how="left")
tabela["codigo_uf_ibge"] = tabela["codigo_municipio"].str[:2]

# Reordena as colunas para organização final
primeiras = ["codigo_municipio", "municipio", "codigo_uf_ibge", "cultura", "ano"]
resto = [c for c in tabela.columns if c not in primeiras]
tabela = tabela[primeiras + resto].sort_values(CHAVES).reset_index(drop=True)

NameError: name 'credito_anual' is not defined